# Suite STR — Structure

`Structure` (`mbse.Architecture.Systems.Framework.Structure`) defines containers, axes, and the three relations
between them: `Contains(c1, c2)`, hard or soft depending on `c1`; `Organizes(hard, soft)`, which makes a container soft
inside a hard one; and `Orients(axis, soft_container)`. These cases cover the schemas `define` builds, and every
problem `Structure.Check` reports that a `unique` clause cannot.

In [ ]:
from mbse.Schemas.Framework import Builders, Validators
from mbse.Architecture.Systems.Framework import Structure
from support import raises, same


def fresh():
    """A store holding the Structure schemas."""
    store = Builders.Store()
    return store, Structure.define(store)


def container(store, name=None, contents=(), organizer=None, axis=None):
    """A container, named where `name` is given, holding `contents`, organized by `organizer`, along `axis`."""
    def spec(c):
        if name is not None:
            c.name(name)
        c.contents(lambda a: [a.insert(lambda e, m=m: e.c2(m)) for m in contents])
        if organizer is not None:
            c.organizer(lambda a: a.insert(lambda e: e.hard(organizer)))
        if axis is not None:
            c.axis(lambda a: a.insert(lambda e: e.axis(axis)))
    return store.Structure.Container(spec)


def add(store, owner, *members):
    """Puts `members` in `owner`."""
    store.Structure.Container(owner, lambda c: c.contents(lambda a: [a.insert(lambda e, m=m: e.c2(m)) for m in members]))

## STR-01 · `define` builds the Structure schemas once, and refuses a store whose names are taken

In [ ]:
store, S = fresh()
schemas = (S.Container, S.Axis, S.Contains, S.Organizes, S.Orients)
assert [s.name for s in schemas] == ["Structure.Container", "Structure.Axis", "Structure.Contains",
                                     "Structure.Organizes", "Structure.Orients"]
assert all(s.validate() == [] for s in schemas)
assert all(store.registered(s.name) is s for s in schemas)
assert list(S.Container.properties) == ["name", "description"] and list(S.Axis.properties) == ["name", "description"]
assert list(S.Container.adjacencies) == ["contents", "containers", "organizes", "organizer", "axis"]
assert list(S.Axis.adjacencies) == ["orients"]
assert all(s.description for s in schemas)

# Defined again, the store gives the schemas it holds.
again = Structure.define(store)
assert all(a is b for a, b in zip(schemas, (again.Container, again.Axis, again.Contains, again.Organizes, again.Orients)))

# A store that holds some of the names, but not all, is refused: they are taken.
other = Builders.Store()
with other.transaction():
    other.Schemas.OfObject(lambda o: o.name("Structure.Axis").ref(True))
with raises(ValueError, match="but not Structure.Container, Structure.Contains, Structure.Organizes, Structure.Orients"):
    Structure.define(other)

## STR-02 · Hard containment: one hard container at most, and none contains itself

In [ ]:
store, S = fresh()
with store.transaction():
    cpu = container(store, "cpu")
    board = container(store, "board", [cpu])
    add(store, board, cpu)  # Contains is a set: containing twice is containing once
assert same([e.c2 for e in board.contents], [cpu]) and same([e.c1 for e in cpu.containers], [board])
assert same(Structure.hard_containers(cpu), [board]) and Structure.hard_containers(board) == []
assert not Structure.is_soft(board) and Structure.organizer(board) is None
assert Structure.Check()(board) == []

# A second hard container.
with store.transaction():
    rack = container(store, "rack", [cpu])
assert Structure.Check()(board) == [
    "Structure.Container 'cpu' has 2 hard containers, Structure.Container 'board', Structure.Container 'rack': an object "
    "has at most one",
]

# A cycle: each container on it contains itself; one hard-contained by a container on it does not.
with store.transaction():
    store.Structure.Container(rack, lambda c: c.contents(lambda a: a.remove(lambda e: e.c2(cpu))))
    add(store, cpu, board)
    chip = container(store, "chip")
    add(store, board, chip)
assert Structure.Check()(board) == [
    "Structure.Container 'board' contains itself, hard",
    "Structure.Container 'cpu' contains itself, hard",
]
assert same(Structure.hard_containers(chip), [board])
assert Validators.Validate(store).Reachable(S.Container, board) == []  # no unique clause sees any of it

## STR-03 · Soft containment: inside the one hard container that organizes it

In [ ]:
store, S = fresh()
with store.transaction():
    desk, lamp = container(store, "desk"), container(store, "lamp")
    floor = container(store, "floor", [desk, lamp])
    corner = container(store, "corner", [desk, lamp], organizer=floor)
    lit = container(store, "lit", [lamp], organizer=floor)  # an object may lie in several soft containers
assert Structure.is_soft(corner) and Structure.organizer(corner) is floor
assert same([e.soft for e in floor.organizes], [corner, lit])
assert same(Structure.hard_containers(lamp), [floor])  # a soft container is no hard container
assert Structure.Check()(floor) == []

# A soft container may lie in another of the same hard container.
with store.transaction():
    nook = container(store, "nook", organizer=floor)
    add(store, corner, nook)
assert Structure.Check()(floor) == []

# What a soft container holds, its hard container must hold too.
with store.transaction():
    chair = container(store, "chair")
    add(store, corner, chair)
assert Structure.Check()(floor) == [
    "Structure.Container 'corner' contains Structure.Container 'chair' softly, but its hard container, "
    "Structure.Container 'floor', does not contain it",
]

# A soft container lies in one hard container: Organizes' unique clause, which mbse-schemas validates.
with store.transaction():
    hall = container(store, "hall")
    store.Structure.Container(corner, lambda c: c.organizer(lambda a: a.insert(lambda e: e.hard(hall))))
assert any(p.startswith("unique(hard) violated") for p in Validators.Validate(store).Reachable(S.Container, floor))

## STR-04 · Axes orient soft containers

In [ ]:
store, S = fresh()
with store.transaction():
    rows, columns = store.Structure.Axis(lambda a: a.name("rows")), store.Structure.Axis(lambda a: a.name("columns"))
    a1 = container(store, "A1")
    sheet = container(store, "sheet", [a1])
    row1 = container(store, "1", [a1], organizer=sheet, axis=rows)
    column_a = container(store, "A", [a1], organizer=sheet, axis=columns)
assert same([e.soft_container for e in rows.orients], [row1]) and same([e.axis for e in column_a.axis], [columns])
assert Structure.Check()(sheet) == []
assert Validators.Validate(store).Reachable(S.Container, sheet) == []

# Only a soft container lies along an axis.
with store.transaction():
    container(store, "loose", axis=rows)
assert Structure.Check()(sheet) == [
    "Structure.Container 'loose' lies along an axis, but no container organizes it: only a soft container lies along an "
    "axis",
]

# A soft container lies along one axis: Orients' unique clause.
with store.transaction():
    store.Structure.Container(row1, lambda c: c.axis(lambda a: a.insert(lambda e: e.axis(columns))))
assert any(p.startswith("unique(axis) violated") for p in Validators.Validate(store).Reachable(S.Container, sheet))

## STR-05 · Problems name an unnamed object by its place among the objects reached

In [ ]:
store, S = fresh()
with store.transaction():
    part = container(store)
    first = container(store, contents=[part])
    second = container(store, contents=[part])
assert Structure.Check()(first) == [
    "Structure.Container #1 has 2 hard containers, Structure.Container #0, Structure.Container #2: an object has at most "
    "one",
]
assert Structure.Check()(part) == [
    "Structure.Container #0 has 2 hard containers, Structure.Container #1, Structure.Container #2: an object has at most "
    "one",
]